# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [62]:
import pandas as pd
import numpy as np

from patsy import dmatrices
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

In [31]:
df = pd.read_csv('previsao_de_renda.csv')

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

In [35]:
df = df.drop(columns = ['Unnamed: 0', 'data_ref', 'id_cliente'])

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   sexo                   15000 non-null  object 
 1   posse_de_veiculo       15000 non-null  bool   
 2   posse_de_imovel        15000 non-null  bool   
 3   qtd_filhos             15000 non-null  int64  
 4   tipo_renda             15000 non-null  object 
 5   educacao               15000 non-null  object 
 6   estado_civil           15000 non-null  object 
 7   tipo_residencia        15000 non-null  object 
 8   idade                  15000 non-null  int64  
 9   tempo_emprego          12427 non-null  float64
 10  qt_pessoas_residencia  15000 non-null  float64
 11  renda                  15000 non-null  float64
dtypes: bool(2), float64(3), int64(2), object(5)
memory usage: 1.2+ MB


1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [50]:
# 1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
X = df.drop(columns=['renda'])
y = df['renda']



# Fórmula: inclua as variáveis que deseja transformar
formula = 'renda ~ C(sexo) + C(estado_civil) + C(tipo_renda) + C(educacao) + C(tipo_residencia)'

# Criando matrizes X (variáveis explicativas) e y (variável dependente)
y, X = dmatrices(formula, data=df, return_type='dataframe')

# Removendo a coluna de resposta (renda) de X, pois já está em y
X = X.drop(columns=['Intercept'])  # Opcional: remova o intercepto, se necessário

# Dividindo em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [52]:
# 2. Rode uma regularização ridge com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o R² na base de testes. Qual o melhor modelo?
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]
ridge_results = {}

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    ridge_results[alpha] = r2_score(y_test, y_pred)

best_ridge_alpha = max(ridge_results, key=ridge_results.get)
print("Melhor alpha (Ridge):", best_ridge_alpha, "com R²:", ridge_results[best_ridge_alpha])


Melhor alpha (Ridge): 0.1 com R²: 0.09571929212526409


In [56]:
lasso_results = {}

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_test)
    lasso_results[alpha] = r2_score(y_test, y_pred)

best_lasso_alpha = max(lasso_results, key=lasso_results.get)
print("Melhor alpha (LASSO):", best_lasso_alpha, "com R²:", lasso_results[best_lasso_alpha])


C:\Anaconda\Lib\site-packages\sklearn\base.py:1474: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
C:\Anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
C:\Anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.509e+11, tolerance: 7.723e+07 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Melhor alpha (LASSO): 0.1 com R²: 0.09572700483041496


In [58]:
lr = LinearRegression()
sfs = SequentialFeatureSelector(lr, direction='forward')
sfs.fit(X_train, y_train)

selected_features = X.columns[sfs.get_support()]
print("Variáveis selecionadas:", selected_features)

Variáveis selecionadas: Index(['C(sexo)[T.M]', 'C(estado_civil)[T.União]',
       'C(tipo_renda)[T.Pensionista]', 'C(tipo_renda)[T.Servidor público]',
       'C(educacao)[T.Superior completo]',
       'C(educacao)[T.Superior incompleto]',
       'C(tipo_residencia)[T.Com os pais]',
       'C(tipo_residencia)[T.Comunitário]', 'C(tipo_residencia)[T.Estúdio]'],
      dtype='object')


#5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?

Comparando os resultados do modelo Ridge e Lasso, não houve grande diferença em relação ao R². Se for considerar a maior simplicidade e interpretabilidade ou o stepwise, onde pode ir adicionando as variáveis até obter o melhor modelo. Mas se o objetivo for um desempenho consistente, o ridge poderia ser mais adequado para o cenário. Eu acredito que ainda criar modelos e ir ajustando variável por variável seja mais simples de interpretar.

In [80]:
# 6. Partindo dos modelos que você ajustou, tente melhorar o na base de testes. 
#    Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.

# Carregar e preparar os dados
df = pd.read_csv('previsao_de_renda.csv')

# Criando novas variáveis transformadas
df['log_idade'] = np.log1p(df['idade'])  # Log transformado
df['sqrt_tempo_emprego'] = np.sqrt(df['tempo_emprego'])  # Raiz quadrada

# Criando variáveis de interação
df['tempo_emprego_estado_civil'] = df['tempo_emprego'] * (df['estado_civil'] == 'Casado')

# Fórmula com variáveis transformadas e interações
formula = 'renda ~ log_idade + sqrt_tempo_emprego + tempo_emprego_estado_civil + C(sexo) + C(estado_civil) + C(tipo_renda) + C(educacao) + C(tipo_residencia)'

# Criando matrizes com patsy
y, X = dmatrices(formula, data=df, return_type='dataframe')

# Dividindo em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Padronizando as variáveis (opcional, mas útil para Ridge e LASSO)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Rodando o modelo Ridge
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]
ridge_results = {}

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    ridge_results[alpha] = r2_score(y_test, y_pred)

best_ridge_alpha = max(ridge_results, key=ridge_results.get)
print("Melhor alpha (Ridge):", best_ridge_alpha, "com R²:", ridge_results[best_ridge_alpha])

# Rodando o modelo LASSO
lasso_results = {}

for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_test)
    lasso_results[alpha] = r2_score(y_test, y_pred)

best_lasso_alpha = max(lasso_results, key=lasso_results.get)
print("Melhor alpha (LASSO):", best_lasso_alpha, "com R²:", lasso_results[best_lasso_alpha])


Melhor alpha (Ridge): 0.1 com R²: 0.29224061663990897


C:\Anaconda\Lib\site-packages\sklearn\base.py:1474: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
C:\Anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
C:\Anaconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.053e+11, tolerance: 8.060e+07 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Melhor alpha (LASSO): 0.1 com R²: 0.2922523368453066


In [82]:
# 7. Ajuste uma árvore de regressão e veja se consegue um R² melhor com ela.

from sklearn.tree import DecisionTreeRegressor

# Ajustando a árvore de regressão
tree = DecisionTreeRegressor(max_depth=5, random_state=42)  # Ajuste max_depth conforme necessário
tree.fit(X_train, y_train)

# Avaliando o R²
y_pred_tree = tree.predict(X_test)
tree_r2 = r2_score(y_test, y_pred_tree)
print("R² (Árvore de Regressão):", tree_r2)


R² (Árvore de Regressão): 0.3517730638184776


Consegui um melhor R² ao ajustar a base de testes e depois realizar a árvore de regressão, isso aumentou o valor do R².

In [84]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid = GridSearchCV(DecisionTreeRegressor(), param_grid, cv=5)
grid.fit(X_train, y_train)
print("Melhores parâmetros:", grid.best_params_)


Melhores parâmetros: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2}
